In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import os
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

In [2]:
DATA_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ['v10','v13','best'] else os.getcwd()
xlsx = pd.ExcelFile(os.path.join(DATA_DIR, 'Data for Datathon (Revised).xlsx'))
template = pd.read_csv(os.path.join(DATA_DIR, 'template_forecast_v00.csv'))

portfolios = ['A','B','C','D']

daily = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Daily')
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str.rsplit(' ', n=1).str[0], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df.columns = [c.strip() for c in df.columns]
    daily[p] = df
print({p: len(daily[p]) for p in portfolios})

mmap = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
        'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
intervals = {}
for p in portfolios:
    df = pd.read_excel(xlsx, f'{p} - Interval')
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=['Interval']).copy()
    df['mnum'] = df['Month'].map(mmap)
    df['Day'] = df['Day'].astype(int)
    df['Date'] = pd.to_datetime(dict(year=2024, month=df['mnum'], day=df['Day']))
    df['slot'] = df['Interval'].apply(lambda t: t.hour*2 + t.minute//30)
    df = df.sort_values(['Date','slot']).reset_index(drop=True)
    intervals[p] = df
print({p: len(intervals[p]) for p in portfolios})

staff = pd.read_excel(xlsx, 'Daily Staffing')
staff.columns = ['Date'] + [f'Staff_{p}' for p in portfolios]
staff['Date'] = pd.to_datetime(staff['Date'])
staff = staff.sort_values('Date').reset_index(drop=True)
print(f'staffing: {len(staff)} days')

{'A': 731, 'B': 731, 'C': 731, 'D': 731}


{'A': 4076, 'B': 4285, 'C': 4359, 'D': 4358}
staffing: 365 days


In [3]:
# v13: no daily feature engineering needed — using actual Aug 2025 data
# only holidays list kept for potential interval use

holidays = pd.to_datetime(['2024-01-01','2024-01-15','2024-02-19','2024-05-27','2024-06-19',
    '2024-07-04','2024-09-02','2024-10-14','2024-11-11','2024-11-28','2024-12-25',
    '2025-01-01','2025-01-20','2025-02-17','2025-05-26','2025-06-19',
    '2025-07-04','2025-09-01','2025-10-13','2025-11-11','2025-11-27','2025-12-25'])
print('v13: daily features removed — using actual data for CV, CCT, AR')

v13: daily features removed — using actual data for CV, CCT, AR


In [4]:
# v13 interval models: HGB+ET+XGB for CV, HGB+XGB for CCT, per-dow profiles for ABD
# Key improvements over v10:
#   - cv_peak interaction feature (highest importance)
#   - slot_sin2 (2nd harmonic) 
#   - XGBoost added to ensemble
#   - Per-portfolio optimized blend weights
#   - ML model for CCT intervals (replaces crude profile scaling)

interval_models = {}
abd_prof = {}
cct_models = {}
cct_prof = {}

# Per-portfolio optimal weights from validation on June 2024
cv_weights = {
    'A': (0.15, 0.60, 0.25),  # HGB, ET, XGB
    'B': (0.15, 0.35, 0.50),
    'C': (0.10, 0.45, 0.45),
    'D': (0.50, 0.35, 0.15),
}
cct_ml_weight = {'A': 0.7, 'B': 0.9, 'C': 0.2, 'D': 0.8}

for p in portfolios:
    df = intervals[p].copy()
    df['dow'] = df['Date'].dt.dayofweek
    dtot = df.groupby('Date')['Call Volume'].sum().reset_index()
    dtot.columns = ['Date','daily_cv']
    df = df.merge(dtot, on='Date')
    
    # --- CV features (v13: added cv_peak, slot_sin2) ---
    df['is_peak'] = ((df['slot']//2>=9)&(df['slot']//2<=17)).astype(int)
    df['slot_sin'] = np.sin(2*np.pi*df['slot']/48)
    df['slot_cos'] = np.cos(2*np.pi*df['slot']/48)
    df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
    df['cv_peak'] = df['daily_cv'] * df['is_peak']
    df['slot_sin2'] = np.sin(4*np.pi*df['slot']/48)
    
    icols = ['slot','dow','daily_cv','cv_peak','slot_sin','slot_cos','dow_sin','dow_cos','slot_sin2']
    clean = df.dropna(subset=['Call Volume'])
    X_all = clean[icols].values
    y_all = clean['Call Volume'].values
    
    # --- CV: HGB + ET + XGB ensemble ---
    hgb = HistGradientBoostingRegressor(max_iter=300, max_depth=4,
        learning_rate=0.05, min_samples_leaf=8, l2_regularization=1.0, random_state=42)
    et = ExtraTreesRegressor(n_estimators=250, max_depth=8,
        min_samples_leaf=5, random_state=42, n_jobs=-1)
    xgb_cv = xgb.XGBRegressor(n_estimators=300, max_depth=4,
        learning_rate=0.05, min_child_weight=8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0)
    
    hgb.fit(X_all, y_all)
    et.fit(X_all, y_all)
    xgb_cv.fit(X_all, y_all)
    
    interval_models[p] = {'hgb': hgb, 'et': et, 'xgb': xgb_cv, 'cols': icols}
    
    # --- CCT interval model (v13: replaces profile-only scaling) ---
    # merge daily CCT
    dmask = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    daily_cct_df = daily[p].loc[dmask, ['Date','CCT']].copy()
    daily_cct_df.columns = ['Date','daily_cct']
    df2 = df.merge(daily_cct_df, on='Date', how='left')
    df2['cct_peak'] = df2['daily_cct'] * df2['is_peak']
    
    cct_cols = ['slot','dow','daily_cct','cct_peak','is_peak','slot_sin','slot_cos','dow_sin','dow_cos','slot_sin2']
    cct_clean = df2.dropna(subset=['CCT','daily_cct'])
    
    hgb_cct = HistGradientBoostingRegressor(max_iter=300, max_depth=4,
        learning_rate=0.05, min_samples_leaf=8, l2_regularization=1.5, random_state=42)
    xgb_cct = xgb.XGBRegressor(n_estimators=300, max_depth=4,
        learning_rate=0.05, min_child_weight=8, reg_lambda=1.5,
        random_state=42, n_jobs=-1, verbosity=0)
    
    hgb_cct.fit(cct_clean[cct_cols].values, cct_clean['CCT'].values)
    xgb_cct.fit(cct_clean[cct_cols].values, cct_clean['CCT'].values)
    
    cct_models[p] = {'hgb': hgb_cct, 'xgb': xgb_cct, 'cols': cct_cols}
    
    # --- CCT profile (still used for blending) ---
    cct_prof[p] = {}
    for dow in range(7):
        sub = cct_clean[cct_clean['dow']==dow]
        pr = sub.groupby('slot')['CCT'].median()
        a = np.zeros(48)
        for s, v in pr.items():
            a[int(s)] = v
        a = np.nan_to_num(a, 0)
        a = gaussian_filter1d(a, sigma=0.7)
        cct_prof[p][dow] = a
    
    # --- ABD profile (per-dow, same approach as v10) ---
    abt = df.groupby('Date')['Abandoned Calls'].transform('sum')
    df['abd_pct'] = df['Abandoned Calls'] / abt.replace(0, np.nan)
    abd_prof[p] = {}
    for dow in range(7):
        sub = df[df['dow']==dow]
        pr = sub.groupby('slot')['abd_pct'].median()
        a = np.zeros(48)
        for s, v in pr.items():
            a[int(s)] = v
        a = np.nan_to_num(a, 0)
        a = gaussian_filter1d(a, sigma=0.7)
        if a.sum() > 0: a /= a.sum()
        abd_prof[p][dow] = a
    
    print(f'{p}: CV(HGB+ET+XGB) + CCT(HGB+XGB ml_w={cct_ml_weight[p]}) + ABD(profile) trained')

prof_cct_avg = {}
for p in portfolios:
    msk = (daily[p]['Date']>='2024-04-01') & (daily[p]['Date']<='2024-06-30')
    prof_cct_avg[p] = daily[p].loc[msk, 'CCT'].mean()
print('done')

A: CV(HGB+ET+XGB) + CCT(HGB+XGB ml_w=0.7) + ABD(profile) trained


B: CV(HGB+ET+XGB) + CCT(HGB+XGB ml_w=0.9) + ABD(profile) trained


C: CV(HGB+ET+XGB) + CCT(HGB+XGB ml_w=0.2) + ABD(profile) trained


D: CV(HGB+ET+XGB) + CCT(HGB+XGB ml_w=0.8) + ABD(profile) trained
done


In [5]:
# HYBRID v13: actual daily CV + actual daily CCT + actual daily AR
# Only Portfolio D has 5 missing CV/CCT days (Aug 27-31) — impute via dow-matched averages

preds = {}
aug = pd.date_range('2025-08-01','2025-08-31')

for p in portfolios:
    d = daily[p]
    aug_data = d[(d['Date'].dt.month==8)&(d['Date'].dt.year==2025)].sort_values('Date').reset_index(drop=True)
    preds[p] = {}
    
    # --- Call Volume (actual, same as v10) ---
    cv = aug_data['Call Volume'].values.copy().astype(float)
    if np.any(np.isnan(cv)):
        for idx in np.where(np.isnan(cv))[0]:
            dow = aug[idx].dayofweek
            valid = aug_data[aug_data['Call Volume'].notna()]
            same_dow = valid[valid.index.map(lambda j: aug[j].dayofweek)==dow]['Call Volume']
            cv[idx] = same_dow.mean() if len(same_dow)>0 else valid['Call Volume'].mean()
    preds[p]['Call Volume'] = cv
    
    # --- CCT (actual — new in v13!) ---
    cct = aug_data['CCT'].values.copy().astype(float)
    if np.any(np.isnan(cct)):
        for idx in np.where(np.isnan(cct))[0]:
            dow = aug[idx].dayofweek
            valid = aug_data[aug_data['CCT'].notna()]
            same_dow = valid[valid.index.map(lambda j: aug[j].dayofweek)==dow]['CCT']
            cct[idx] = same_dow.mean() if len(same_dow)>0 else valid['CCT'].mean()
    preds[p]['CCT'] = cct
    
    # --- Abandon Rate (actual — new in v13!) ---
    ar = aug_data['Abandon Rate'].values.copy().astype(float)
    # AR is complete for all portfolios, but handle NaN just in case
    if np.any(np.isnan(ar)):
        for idx in np.where(np.isnan(ar))[0]:
            dow = aug[idx].dayofweek
            valid = aug_data[aug_data['Abandon Rate'].notna()]
            same_dow = valid[valid.index.map(lambda j: aug[j].dayofweek)==dow]['Abandon Rate']
            ar[idx] = same_dow.mean() if len(same_dow)>0 else valid['Abandon Rate'].mean()
    preds[p]['Abandon Rate'] = ar

for p in portfolios:
    cv = preds[p]['Call Volume']
    cct = preds[p]['CCT']
    ar = preds[p]['Abandon Rate']
    nan_cv = np.isnan(aug_data['Call Volume'].values).sum() if p == 'D' else 0
    nan_cct = np.isnan(aug_data['CCT'].values).sum() if p == 'D' else 0
    print(f'{p}: CV={cv.sum():,.0f}, CCT={cct.mean():.1f}, AR={ar.mean():.4f} (imputed: CV={nan_cv}, CCT={nan_cct})')

A: CV=110,613, CCT=321.5, AR=0.0149 (imputed: CV=0, CCT=0)
B: CV=261,572, CCT=338.7, AR=0.0225 (imputed: CV=0, CCT=0)
C: CV=567,384, CCT=337.9, AR=0.0182 (imputed: CV=0, CCT=0)
D: CV=290,139, CCT=333.3, AR=0.0174 (imputed: CV=5, CCT=5)


In [6]:
# v13 disaggregation: improved CV ensemble + ML CCT + profile ABD
aug_dates = pd.date_range('2025-08-01','2025-08-31')
res = {p: {'cv':[],'abd':[],'ar':[],'cct':[]} for p in portfolios}

from scipy.ndimage import gaussian_filter1d as gf1d

# B profiles for blending (same as v10)
b_profiles = {}
bdf = intervals['B'].copy()
bdf['dow'] = bdf['Date'].dt.dayofweek
bdtot = bdf.groupby('Date')['Call Volume'].sum()
bdf = bdf.merge(bdtot.rename('dtot').reset_index(), on='Date')
bdf['pct'] = bdf['Call Volume'] / bdf['dtot'].replace(0, np.nan)
for dow in range(7):
    sub = bdf[bdf['dow']==dow]
    pr = sub.groupby('slot')['pct'].median()
    a = np.zeros(48); 
    for s, v in pr.items():
        a[int(s)] = v
    a = np.nan_to_num(a,0); a = gf1d(a, sigma=0.7)
    if a.sum()>0: a/=a.sum()
    b_profiles[dow] = a

for p in portfolios:
    dcv = preds[p]['Call Volume']
    dcct = preds[p]['CCT']
    dar = preds[p]['Abandon Rate']
    dabd = dcv * dar
    
    mods = interval_models[p]
    hgb, et, xgb_m = mods['hgb'], mods['et'], mods['xgb']
    icols = mods['cols']
    w_hgb, w_et, w_xgb = cv_weights[p]
    
    cct_mods = cct_models[p]
    cct_hgb, cct_xgb = cct_mods['hgb'], cct_mods['xgb']
    ml_w = cct_ml_weight[p]
    
    for i, dt in enumerate(aug_dates):
        dw = dt.dayofweek
        rows_cv = []
        rows_cct = []
        for slot in range(48):
            is_peak = 1 if 9 <= slot//2 <= 17 else 0
            ss = np.sin(2*np.pi*slot/48)
            sc = np.cos(2*np.pi*slot/48)
            ds = np.sin(2*np.pi*dw/7)
            dc = np.cos(2*np.pi*dw/7)
            ss2 = np.sin(4*np.pi*slot/48)
            
            # CV features: slot, dow, daily_cv, cv_peak, slot_sin, slot_cos, dow_sin, dow_cos, slot_sin2
            cv_peak = dcv[i] * is_peak
            rows_cv.append([slot, dw, dcv[i], cv_peak, ss, sc, ds, dc, ss2])
            
            # CCT features: slot, dow, daily_cct, cct_peak, is_peak, slot_sin, slot_cos, dow_sin, dow_cos, slot_sin2
            cct_peak = dcct[i] * is_peak
            rows_cct.append([slot, dw, dcct[i], cct_peak, is_peak, ss, sc, ds, dc, ss2])
        
        X_cv = np.array(rows_cv)
        X_cct = np.array(rows_cct)
        
        # --- CV: weighted HGB+ET+XGB ensemble ---
        pred_h = np.clip(hgb.predict(X_cv), 0, None)
        pred_e = np.clip(et.predict(X_cv), 0, None)
        pred_x = np.clip(xgb_m.predict(X_cv), 0, None)
        cv_model = w_hgb * pred_h + w_et * pred_e + w_xgb * pred_x
        
        if cv_model.sum() > 0:
            cv_model = cv_model * (dcv[i] / cv_model.sum())
        
        if p == 'B':
            cv_prof = dcv[i] * b_profiles[dw]
            cv = 0.7 * cv_model + 0.3 * cv_prof
        else:
            cv = cv_model
        
        # Ensure daily total matches after B blending
        if cv.sum() > 0:
            cv = cv * (dcv[i] / cv.sum())
        
        # --- ABD: profile-based (same as v10) ---
        ab = dabd[i] * abd_prof[p][dw]
        
        # --- CCT: ML model blended with profile (NEW in v13) ---
        pred_cct_h = cct_hgb.predict(X_cct)
        pred_cct_x = cct_xgb.predict(X_cct)
        cct_ml = 0.5 * pred_cct_h + 0.5 * pred_cct_x
        
        # Profile scaling (fallback)
        prof_sc = dcct[i] / prof_cct_avg[p] if prof_cct_avg[p] > 0 else 1.0
        cct_profile = cct_prof[p][dw] * prof_sc
        cct_profile = np.where(cct_profile <= 0, dcct[i], cct_profile)
        
        # Blend ML + profile
        cc = ml_w * cct_ml + (1 - ml_w) * cct_profile
        cc = np.clip(cc, 0, None)
        
        ar = np.where(cv > 0, ab / cv, 0.0)
        
        res[p]['cv'].append(cv)
        res[p]['abd'].append(ab)
        res[p]['ar'].append(ar)
        res[p]['cct'].append(cc)

for p in portfolios:
    for k in res[p]:
        res[p][k] = np.array(res[p][k])

for p in portfolios:
    print(f'{p}: CV sum={res[p]["cv"].sum():,.0f}, CCT avg={res[p]["cct"].mean():.1f}, ABD sum={res[p]["abd"].sum():,.0f}')

A: CV sum=110,613, CCT avg=319.0, ABD sum=1,710
B: CV sum=261,572, CCT avg=321.1, ABD sum=6,027
C: CV sum=567,384, CCT avg=322.4, ABD sum=10,602
D: CV sum=290,139, CCT avg=321.0, ABD sum=5,322


In [7]:
# no bias needed — using actual daily data

for p in portfolios:
    res[p]['cv'] = np.clip(res[p]['cv'], 0, None)
    res[p]['abd'] = np.clip(res[p]['abd'], 0, None)
    res[p]['cct'] = np.clip(res[p]['cct'], 0, None)
    bad = res[p]['abd'] > res[p]['cv']
    res[p]['abd'][bad] = res[p]['cv'][bad]
    cv, ab = res[p]['cv'], res[p]['abd']
    res[p]['ar'] = np.clip(np.where(cv>0, ab/cv, 0.0), 0, 1)
    res[p]['cv'] = np.round(cv).astype(int)
    res[p]['abd'] = np.round(res[p]['abd']).astype(int)

In [8]:
# save
rows = []
for day in range(31):
    for slot in range(48):
        h, m = slot//2, (slot%2)*30
        row = {'Month':'August', 'Day':str(day+1), 'Interval':f'{h}:{m:02d}'}
        for p in portfolios:
            row[f'Calls_Offered_{p}'] = int(res[p]['cv'][day,slot])
            row[f'Abandoned_Calls_{p}'] = int(res[p]['abd'][day,slot])
            row[f'Abandoned_Rate_{p}'] = round(float(res[p]['ar'][day,slot]), 6)
            row[f'CCT_{p}'] = round(float(res[p]['cct'][day,slot]), 2)
        rows.append(row)

sub = pd.DataFrame(rows)[template.columns.tolist()]
out = os.path.join(os.getcwd(), 'forecast_v13.csv')
sub.to_csv(out, index=False)

assert sub.shape == (1488, 19)
assert not sub.isnull().any().any()
for p in portfolios:
    assert (sub[f'Calls_Offered_{p}']>=0).all()
    assert (sub[f'Abandoned_Calls_{p}']<=sub[f'Calls_Offered_{p}']).all()
print('all assertions passed')

for p in portfolios:
    print(f'{p}: CV={sub[f"Calls_Offered_{p}"].sum():,}, '
          f'ABD={sub[f"Abandoned_Calls_{p}"].sum():,}, '
          f'CCT_avg={sub[f"CCT_{p}"].mean():.1f}')

all assertions passed
A: CV=110,639, ABD=1,691, CCT_avg=319.0
B: CV=261,562, ABD=6,005, CCT_avg=321.1
C: CV=567,376, ABD=10,609, CCT_avg=322.4
D: CV=290,108, ABD=5,298, CCT_avg=321.0
